# 07a. Frequency-Domain Features & Correlation Analysis

## Objective

This notebook extracts frequency-domain wearable features from the preprocessed Parkinson's Disease Smartwatch Dataset (PADS) signals and identifies strongly correlated features.

Following the decision established in Notebook 04, each post-trim recording is first interpolated onto a **uniform 100 Hz grid** before FFT-based feature extraction. The interpolation preserves the established post-trim recording lengths of **976** and **2,000 samples**.

The notebook produces:

- a frequency-domain feature table;
- accelerometer and gyroscope frequency-feature groups;
- a Pearson correlation matrix;
- a summary of strongly correlated wearable features; and
- validation confirming that uniform resampling preserves the post-trim sample lengths.


## Frequency-Domain Features

For each accelerometer and gyroscope axis and magnitude signal, the following features are extracted:

- **Dominant frequency**
- **Spectral centroid**
- **Spectral entropy**
- **Spectral power**

The analysis includes accelerometer X, Y, Z, and magnitude, together with gyroscope X, Y, Z, and magnitude. This produces **32 frequency-domain features per recording**.


## Uniform 100 Hz Resampling

Notebook 04 found timestamp jitter in the raw smartwatch recordings and concluded that FFT-based features should be calculated only after interpolation onto a uniform 100 Hz grid.

The feature module therefore:

1. reads the post-trim preprocessed recordings;
2. replaces irregular timestamps with exact 0.01-second spacing;
3. linearly interpolates all accelerometer, gyroscope, and magnitude channels;
4. preserves the existing number of samples; and
5. performs the FFT using a fixed sampling frequency of 100 Hz.

The original timestamp-based sampling frequency is retained only as a quality-assurance field. It is not used for the FFT.


In [ ]:
# Libraries and project paths
# =============================================================================

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root containing data/ and src/."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.features.frequency_features import (
    TARGET_SAMPLING_FREQUENCY,
    build_correlation_summary,
    build_frequency_feature_table,
    compute_feature_correlations,
    get_accelerometer_feature_columns,
    get_frequency_feature_columns,
    get_gyroscope_feature_columns,
    summarize_strong_correlations,
)

NPZ_DIR = PROJECT_ROOT / "data" / "processed" / "preprocessed_signals"

FEATURE_OUTPUT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "frequency_domain_features.csv"
)

CORRELATION_OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "tables"
    / "correlation_analysis"
)

CORRELATION_MATRIX_OUTPUT = (
    CORRELATION_OUTPUT_DIR
    / "frequency_feature_correlation_matrix.csv"
)

STRONG_CORRELATIONS_OUTPUT = (
    CORRELATION_OUTPUT_DIR
    / "strongly_correlated_features.csv"
)

CORRELATION_SUMMARY_OUTPUT = (
    CORRELATION_OUTPUT_DIR
    / "correlation_analysis_summary.csv"
)

RESAMPLING_LENGTH_OUTPUT = (
    CORRELATION_OUTPUT_DIR
    / "post_resampling_length_summary.csv"
)

STRONG_CORRELATION_THRESHOLD = 0.80
EXPECTED_POST_TRIM_LENGTHS = {976, 2000}

print("Project root:", PROJECT_ROOT)
print("Processed signal directory:", NPZ_DIR)
print("Target sampling frequency:", TARGET_SAMPLING_FREQUENCY, "Hz")


## 1. Confirm Input Availability

In [ ]:
if not NPZ_DIR.exists():
    raise FileNotFoundError(
        f"Could not locate {NPZ_DIR}. "
        "Run or obtain the signal-preprocessing outputs first."
    )

npz_files = sorted(NPZ_DIR.glob("*_preprocessed.npz"))

if not npz_files:
    raise FileNotFoundError(
        f"No participant .npz files were found in {NPZ_DIR}."
    )

print("Participant NPZ files found:", len(npz_files))
print("First file:", npz_files[0].name)

assert len(npz_files) == 469, (
    f"Expected 469 participant files, found {len(npz_files)}."
)


## 2. Inspect the Preprocessed Signal Structure

In [ ]:
with np.load(npz_files[0], allow_pickle=False) as sample:
    sample_columns = sample["__columns__"].astype(str).tolist()
    sample_recording_keys = (
        sample["__recording_keys__"].astype(str).tolist()
    )
    first_recording_key = sample_recording_keys[0]
    first_recording_shape = sample[first_recording_key].shape

print("Signal columns:", sample_columns)
print("Recordings in sample participant:", len(sample_recording_keys))
print("First recording key:", first_recording_key)
print("First recording shape:", first_recording_shape)

expected_columns = [
    "Time",
    "Accelerometer_X",
    "Accelerometer_Y",
    "Accelerometer_Z",
    "Gyroscope_X",
    "Gyroscope_Y",
    "Gyroscope_Z",
    "Acc_Magnitude",
    "Gyro_Magnitude",
]

missing_columns = [
    column
    for column in expected_columns
    if column not in sample_columns
]

assert not missing_columns
assert len(sample_recording_keys) == 22


## 3. Extract Frequency-Domain Features After 100 Hz Resampling

In [ ]:
frequency_features = build_frequency_feature_table(NPZ_DIR)

print("Frequency feature table shape:", frequency_features.shape)
print("Participants:", frequency_features["patient_id"].nunique())
print("Recordings:", len(frequency_features))
print("Tasks:", frequency_features["task"].nunique())
print("Wrists:", frequency_features["wrist"].nunique())

assert frequency_features["patient_id"].nunique() == 469
assert len(frequency_features) == 10_318

frequency_features.head()


## 4. Validate Uniform 100 Hz Resampling

These checks confirm that:

- every recording was analyzed at exactly 100 Hz;
- the resampled sample count equals the original post-trim sample count;
- the only retained lengths are 976 and 2,000 samples; and
- the uniform grid starts at 0 seconds and has the expected end time.


In [ ]:
required_resampling_columns = [
    "n_samples",
    "original_sampling_frequency_hz",
    "sampling_frequency_hz",
    "resampled_n_samples",
    "uniform_time_start_s",
    "uniform_time_end_s",
]

missing_resampling_columns = [
    column
    for column in required_resampling_columns
    if column not in frequency_features.columns
]

print("Missing validation columns:", missing_resampling_columns)
assert not missing_resampling_columns

assert np.allclose(
    frequency_features["sampling_frequency_hz"],
    TARGET_SAMPLING_FREQUENCY,
)


In [ ]:
lengths_preserved = (
    frequency_features["n_samples"]
    == frequency_features["resampled_n_samples"]
)

print("Recordings with preserved length:", int(lengths_preserved.sum()))
print("Recordings with changed length:", int((~lengths_preserved).sum()))

assert lengths_preserved.all()


In [ ]:
post_resampling_length_summary = (
    frequency_features["resampled_n_samples"]
    .value_counts()
    .sort_index()
    .rename_axis("post_trim_samples")
    .reset_index(name="recordings")
)

observed_lengths = set(
    post_resampling_length_summary["post_trim_samples"]
    .astype(int)
    .tolist()
)

print("Observed post-resampling lengths:", sorted(observed_lengths))
print("Expected post-trim lengths:", sorted(EXPECTED_POST_TRIM_LENGTHS))

assert observed_lengths == EXPECTED_POST_TRIM_LENGTHS

post_resampling_length_summary


In [ ]:
expected_end_time = (
    frequency_features["resampled_n_samples"] - 1
) / TARGET_SAMPLING_FREQUENCY

valid_uniform_grid = (
    np.isclose(
        frequency_features["uniform_time_start_s"],
        0.0,
    )
    & np.isclose(
        frequency_features["uniform_time_end_s"],
        expected_end_time,
    )
)

print("Recordings with valid 100 Hz grid:", int(valid_uniform_grid.sum()))
print("Recordings with invalid grid:", int((~valid_uniform_grid).sum()))

assert valid_uniform_grid.all()

print(
    "Confirmed: all recordings were resampled onto a uniform "
    "100 Hz grid and retained lengths of 976 or 2,000 samples."
)


## 5. Original Sampling-Frequency QA

The original timestamp-based sampling frequency is included only for quality-assurance reporting. The FFT uses the uniformly resampled 100 Hz signals.


In [ ]:
original_sampling_qa = (
    frequency_features["original_sampling_frequency_hz"]
    .describe()
    .to_frame(name="value")
)

original_sampling_qa


## 6. Accelerometer and Gyroscope Feature Groups

In [ ]:
all_frequency_columns = get_frequency_feature_columns(frequency_features)

accelerometer_frequency_columns = (
    get_accelerometer_feature_columns(frequency_features)
)

gyroscope_frequency_columns = (
    get_gyroscope_feature_columns(frequency_features)
)

feature_group_summary = pd.DataFrame(
    {
        "feature_group": [
            "Accelerometer",
            "Gyroscope",
            "Total",
        ],
        "number_of_features": [
            len(accelerometer_frequency_columns),
            len(gyroscope_frequency_columns),
            len(all_frequency_columns),
        ],
    }
)

assert len(accelerometer_frequency_columns) == 16
assert len(gyroscope_frequency_columns) == 16
assert len(all_frequency_columns) == 32

feature_group_summary


## 7. Frequency-Feature Quality Assurance

In [ ]:
identifier_columns = ["patient_id", "task", "wrist"]

duplicate_recordings = frequency_features.duplicated(
    subset=identifier_columns,
    keep=False,
)

feature_qa = pd.DataFrame(
    {
        "metric": [
            "Rows",
            "Unique participant-task-wrist records",
            "Duplicate participant-task-wrist records",
            "Frequency features",
            "Missing frequency-feature values",
            "Infinite frequency-feature values",
            "Recordings resampled at 100 Hz",
            "Recordings retaining post-trim length",
        ],
        "value": [
            len(frequency_features),
            frequency_features[identifier_columns]
            .drop_duplicates()
            .shape[0],
            int(duplicate_recordings.sum()),
            len(all_frequency_columns),
            int(
                frequency_features[all_frequency_columns]
                .isna()
                .sum()
                .sum()
            ),
            int(
                np.isinf(
                    frequency_features[all_frequency_columns]
                    .to_numpy(dtype=float)
                ).sum()
            ),
            int(
                np.isclose(
                    frequency_features["sampling_frequency_hz"],
                    TARGET_SAMPLING_FREQUENCY,
                ).sum()
            ),
            int(lengths_preserved.sum()),
        ],
    }
)

feature_qa


## 8. Save the Frequency-Domain Feature Table

In [ ]:
FEATURE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)

frequency_features.to_csv(
    FEATURE_OUTPUT,
    index=False,
)

if not FEATURE_OUTPUT.exists() or FEATURE_OUTPUT.stat().st_size == 0:
    raise IOError(
        f"Frequency feature table was not saved correctly: {FEATURE_OUTPUT}"
    )

verified_frequency_features = pd.read_csv(
    FEATURE_OUTPUT,
    dtype={"patient_id": str},
)

assert verified_frequency_features.shape == frequency_features.shape

print("Saved:", FEATURE_OUTPUT)
print("Verified shape:", verified_frequency_features.shape)


## 9. Correlation Analysis

In [ ]:
correlation_matrix = compute_feature_correlations(
    frequency_features,
    method="pearson",
)

print("Correlation matrix shape:", correlation_matrix.shape)
assert correlation_matrix.shape == (32, 32)

correlation_matrix.round(3)


In [ ]:
figure, axis = plt.subplots(figsize=(15, 13))

image = axis.imshow(
    correlation_matrix.to_numpy(),
    aspect="auto",
    vmin=-1,
    vmax=1,
)

axis.set_xticks(range(len(correlation_matrix.columns)))
axis.set_xticklabels(
    correlation_matrix.columns,
    rotation=90,
    fontsize=7,
)

axis.set_yticks(range(len(correlation_matrix.index)))
axis.set_yticklabels(
    correlation_matrix.index,
    fontsize=7,
)

axis.set_title(
    "Frequency-Domain Feature Correlations "
    "(Uniform 100 Hz Signals)"
)

figure.colorbar(
    image,
    ax=axis,
    label="Pearson correlation",
)

figure.tight_layout()
plt.show()


## 10. Strongly Correlated Wearable Features

In [ ]:
strong_correlations = summarize_strong_correlations(
    correlation_matrix,
    threshold=STRONG_CORRELATION_THRESHOLD,
)

print("Strongly correlated pairs:", len(strong_correlations))
strong_correlations.head(30)


In [ ]:
if strong_correlations.empty:
    strong_correlation_group_summary = pd.DataFrame(
        columns=[
            "relationship_group",
            "direction",
            "feature_pairs",
        ]
    )
else:
    strong_correlation_group_summary = (
        strong_correlations
        .groupby(
            ["relationship_group", "direction"],
            as_index=False,
        )
        .size()
        .rename(columns={"size": "feature_pairs"})
    )

strong_correlation_group_summary


## 11. Save Correlation and Resampling Outputs

In [ ]:
correlation_summary = build_correlation_summary(
    feature_table=frequency_features,
    strong_correlations=strong_correlations,
    threshold=STRONG_CORRELATION_THRESHOLD,
)

additional_summary_rows = pd.DataFrame(
    {
        "metric": [
            "Target sampling frequency (Hz)",
            "Recordings with preserved post-trim length",
            "Observed post-trim lengths",
        ],
        "value": [
            TARGET_SAMPLING_FREQUENCY,
            int(lengths_preserved.sum()),
            ", ".join(
                str(value)
                for value in sorted(observed_lengths)
            ),
        ],
    }
)

correlation_summary = pd.concat(
    [correlation_summary, additional_summary_rows],
    ignore_index=True,
)

CORRELATION_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

correlation_matrix.to_csv(CORRELATION_MATRIX_OUTPUT)

strong_correlations.to_csv(
    STRONG_CORRELATIONS_OUTPUT,
    index=False,
)

correlation_summary.to_csv(
    CORRELATION_SUMMARY_OUTPUT,
    index=False,
)

post_resampling_length_summary.to_csv(
    RESAMPLING_LENGTH_OUTPUT,
    index=False,
)

output_paths = [
    CORRELATION_MATRIX_OUTPUT,
    STRONG_CORRELATIONS_OUTPUT,
    CORRELATION_SUMMARY_OUTPUT,
    RESAMPLING_LENGTH_OUTPUT,
]

for output_path in output_paths:
    if not output_path.exists() or output_path.stat().st_size == 0:
        raise IOError(
            f"Output was not saved correctly: {output_path}"
        )
    print("Saved:", output_path)

correlation_summary


## 12. Final Summary

Upon successful execution, this notebook confirms that:

- all 469 participant files were processed;
- 10,318 recordings were analyzed;
- every recording was interpolated onto a uniform 100 Hz grid before FFT-based feature extraction;
- the original timestamp-based sampling frequency was retained only for QA;
- the post-trim lengths of 976 and 2,000 samples were preserved;
- 32 frequency-domain features were generated;
- accelerometer and gyroscope feature groups were created;
- correlation outputs were generated and saved; and
- strongly correlated wearable feature pairs were identified using `|r| ≥ 0.80`.
